# VIDA + PANDA — Multi-Agent VLM Crop Disease Diagnosis
### CDDM Benchmark | Zero-Shot | Named Peer Deliberation

**Framework:**
- **VIDA** (Varied Independent Diagnostic Assessment) — 6 agents independently diagnose each image in Round 1
- **PANDA** (Peer-Anchored Named Deliberation Architecture) — top-3 agents debate in Rounds 2 & 3 with named peer attribution

**Agents (6, 4 providers):**

| Agent | Provider | Model string |
|---|---|---|
| GPT-4.1 | OpenAI | `gpt-4.1` |
| Grok-4.20-Fast | xAI | `grok-4.20-non-reasoning` |
| GPT-4.1-mini | OpenAI | `gpt-4.1-mini` |
| Claude Sonnet 4.5 | Anthropic | `claude-sonnet-4-5` |
| Claude Haiku 4.5 | Anthropic | `claude-haiku-4-5` |
| Gemma-3n-E4B | Together.ai | `google/gemma-3n-E4B-it` |

**Dataset:** CDDM (Liu et al., 2024) — 490 images, 53 classes, stratified pilot  
**Paper:** *A Multi-Agent Vision-Language Debate Framework for Zero-Shot Crop Disease Diagnosis*

---
## Workflow
1. **Cells 1–9**: Setup, agents, prompts, parser
2. **Cell 10**: Run VIDA (Round 1 — all 6 agents independent)
3. **Cell 11**: GPT-4.1 judge scoring
4. **Cell 12**: VIDA metrics dashboard
5. **Cell 13**: Select debate agents (auto top-3 or manual)
6. **Cell 14**: Run PANDA (Rounds 2 + 3 debate)
7. **Cell 15**: Influence analysis + visualization
8. **Cell 16**: Full summary
---

## Cell 1: Install Dependencies

In [ ]:
%%capture
!pip install openai anthropic together pillow pandas matplotlib seaborn tqdm scikit-learn -q

## Cell 2: Imports & API Keys

In [ ]:
import os, json, base64, random, time, re, gc, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter, defaultdict

import openai
import anthropic
from together import Together

# ── Kaggle Secrets ──────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

OPENAI_API_KEY    = secrets.get_secret("OPENAI_API_KEY")
ANTHROPIC_API_KEY = secrets.get_secret("ANTHROPIC_API_KEY")
TOGETHER_API_KEY  = secrets.get_secret("TOGETHER_API_KEY")
XAI_API_KEY       = secrets.get_secret("XAI_API_KEY")   # xAI API key for Grok

openai_client    = openai.OpenAI(api_key=OPENAI_API_KEY)
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
together_client  = Together(api_key=TOGETHER_API_KEY)
# Grok uses OpenAI-compatible API at a different base URL
grok_client      = openai.OpenAI(api_key=XAI_API_KEY, base_url="https://api.x.ai/v1")

print("✅ All API clients initialized")

## Cell 3: CDDM Taxonomy & Dataset Config

In [ ]:
# ── CDDM 54-class taxonomy ──────────────────────────────────────────────────────
# (Tomato Yellow Leaf Curl absent from accessible partition → 53 classes in practice)
CDDM_CLASSES = [
    "Apple,Alternaria Blotch","Apple,Black Rot","Apple,Brown Spot",
    "Apple,Cedar Apple Rust","Apple,Frog Eye Leaf Spot","Apple,Grey Spot",
    "Apple,Healthy","Apple,Leaf Rust","Apple,Mosaic Virus",
    "Apple,Powdery Mildew","Apple,Scab",
    "Bell Pepper,Bacterial Spot","Bell Pepper,Healthy",
    "Blueberry,Healthy","Cherry,Healthy","Cherry,Powdery Mildew",
    "Corn,Healthy","Corn,Leaf Rust","Corn,Leaf Spot","Corn,Northern Leaf Blight",
    "Grape,Black Rot","Grape,Esca","Grape,Healthy","Grape,Leaf Blight",
    "Orange,Citrus Greening","Orange,Healthy",
    "Peach,Bacterial Spot","Peach,Healthy",
    "Potato,Early Blight","Potato,Healthy","Potato,Late Blight",
    "Pumpkin,Powdery Mildew","Raspberry,Healthy",
    "Rice,Bacterial Leaf Blight","Rice,Blast","Rice,Brown Spot",
    "Rice,Leaf Blight","Rice,Leaf Smut","Rice,Tungro",
    "Soybean,Healthy","Strawberry,Healthy","Strawberry,Leaf Scorch",
    "Tomato,Bacterial Spot","Tomato,Early Blight","Tomato,Healthy",
    "Tomato,Late Blight","Tomato,Leaf Mold","Tomato,Mosaic Virus",
    "Tomato,Powdery Mildew","Tomato,Septoria Leaf Spot","Tomato,Spider Mites",
    "Tomato,Target Spot","Tomato,Yellow Leaf Curl","Wheat,Healthy",
]

CDDM_CROPS    = sorted(set(c.split(',')[0].strip() for c in CDDM_CLASSES))
CDDM_DISEASES = sorted(set(c.split(',')[1].strip() for c in CDDM_CLASSES))
HEALTHY_LABEL = "Healthy"
_CROP_LIST    = ', '.join(CDDM_CROPS)
_DISEASE_LIST = ', '.join(CDDM_DISEASES)

# ── Paths ────────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("/kaggle/input/datasets/maljuboori/cddm-dataset/images")
OUTPUT_DIR   = Path("/kaggle/working/vida_panda_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Sampling ─────────────────────────────────────────────────────────────────────
N_IMAGES     = 490    # Target total (≈10 per class, 80% diseased / 20% healthy)
RANDOM_SEED  = 42
N_DEBATE     = 3      # PANDA agents (top-3 by Combined Accuracy)
SOFTMAX_TEMP = 2.0    # Temperature for softmax voting weights
CHECKPOINT   = 10     # Save checkpoint every N images

print(f"✅ {len(CDDM_CLASSES)} classes | {len(CDDM_CROPS)} crops | {len(CDDM_DISEASES)} disease types")
print(f"   Target: {N_IMAGES} images | PANDA agents: {N_DEBATE} | Softmax T={SOFTMAX_TEMP}")

## Cell 4: Load Dataset — Stratified Pilot (490 images)

In [ ]:
random.seed(RANDOM_SEED)

# Build pool: up to ceil(N_IMAGES / n_classes) images per class
all_pairs = []
missing   = []
per_class = max(2, math.ceil(N_IMAGES / len(CDDM_CLASSES)))  # ≈9–10

for cl in CDDM_CLASSES:
    d = DATASET_ROOT / cl
    if not d.exists():
        missing.append(cl); continue
    imgs = sorted(d.glob("*.jpg")) + sorted(d.glob("*.png"))
    if not imgs:
        missing.append(cl); continue
    crop, disease = cl.split(',', 1)
    crop    = crop.strip()
    disease = disease.strip()
    gt_dis  = 0 if disease == HEALTHY_LABEL else 1
    sampled = random.sample(imgs, min(per_class, len(imgs)))
    for img_path in sampled:
        all_pairs.append({
            "image_path":  img_path,
            "class_label": cl,
            "gt_crop":     crop,
            "gt_disease":  disease,
            "gt_diseased": gt_dis,
        })

random.shuffle(all_pairs)

# Stratified trim: ~80% diseased / 20% healthy
diseased_pool = [p for p in all_pairs if p["gt_diseased"] == 1]
healthy_pool  = [p for p in all_pairs if p["gt_diseased"] == 0]
n_healthy  = min(N_IMAGES // 5, len(healthy_pool))
n_diseased = min(N_IMAGES - n_healthy, len(diseased_pool))
pilot = (random.sample(diseased_pool, n_diseased) +
         random.sample(healthy_pool,  n_healthy))
random.shuffle(pilot)

if missing:
    print(f"⚠️  Missing classes ({len(missing)}): {missing}")
print(f"✅ Pilot: {len(pilot)} images")
print(f"   Diseased: {sum(p['gt_diseased'] for p in pilot)}  Healthy: {sum(1-p['gt_diseased'] for p in pilot)}")
print(f"   Classes covered: {len(set(p['class_label'] for p in pilot))}")

## Cell 5: Prompt Templates (R1, R2, R3)

In [ ]:
# ── Round 1: Independent analysis ──────────────────────────────────────────────
ROUND1_PROMPT = (
    "You are a plant science researcher analyzing a crop leaf image. "
    "Identify the crop and any disease present.\n\n"
    "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
    f"Crop Category: <one of: {_CROP_LIST}>\n"
    f"Disease Category: <one of: {_DISEASE_LIST}>\n"
    "Diseased: <Yes or No>\n"
    "Reasoning: <2-3 sentences citing specific visual evidence>\n"
    "Confidence: <0.0 to 1.0>"
)

# ── Round 2 / 3 format blocks ───────────────────────────────────────────────────
_FMT_R2 = (
    "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
    f"Crop Category: <one of: {_CROP_LIST}>\n"
    f"Disease Category: <one of: {_DISEASE_LIST}>\n"
    "Diseased: <Yes or No>\n"
    "Reasoning: <address peers by name, cite visual evidence>\n"
    "Confidence: <0.0 to 1.0>\n"
    "Influenced By: <exact peer name who changed your verdict, or None>\n"
    "New Visual Evidence: <new visual feature you observed, or NONE>"
)
_FMT_R3 = (
    "BEGIN YOUR RESPONSE WITH THESE FIELDS — NO TEXT BEFORE THEM:\n"
    f"Crop Category: <one of: {_CROP_LIST}>\n"
    f"Disease Category: <one of: {_DISEASE_LIST}>\n"
    "Diseased: <Yes or No>\n"
    "Reasoning: <final position, address peers by name>\n"
    "Confidence: <0.0 to 1.0>\n"
    "Influenced By: <exact peer name who changed your verdict vs R2, or None>\n"
    "New Visual Evidence: <new visual feature, or NONE>"
)

def make_r2_prompt(my_name, own_r1, others_r1):
    def s(p): return f"{p.get('crop_category','?')} / {p.get('disease_category','?')}"
    own = (f"YOUR Round 1 verdict: {s(own_r1)}\n"
           f"  Reasoning: {own_r1.get('reasoning','')[:200]}")
    peers = ''
    for pn, p in others_r1.items():
        peers += f"\n[{pn}]: {s(p)}\n  Reasoning: {p.get('reasoning','')[:150]}\n"
    peer_names = list(others_r1.keys())
    return (
        f"You are {my_name}, a plant science researcher.\n\n{own}\n\n"
        f"Peer assessments of the SAME image:\n{peers}\n"
        f"Re-examine the image. Address {' and '.join(peer_names)} by name — "
        f"AGREE or DISAGREE with their crop and disease ID, citing specific visual features. "
        f"Only change your verdict if you identify NEW visual evidence not in your R1 reasoning.\n\n"
        + _FMT_R2
    )

def make_r3_prompt(my_name, own_r1, own_r2, others_r2):
    def s(p): return f"{p.get('crop_category','?')} / {p.get('disease_category','?')}"
    history = (f"YOUR history: R1={s(own_r1)} → R2={s(own_r2)}\n"
               f"  R2 Reasoning: {own_r2.get('reasoning','')[:200]}")
    peers = ''
    for pn, p in others_r2.items():
        peers += f"\n[{pn}] R2: {s(p)}\n  Reasoning: {p.get('reasoning','')[:150]}\n"
    peer_names = list(others_r2.keys())
    return (
        f"You are {my_name}. FINAL ROUND — commit to your verdict.\n\n{history}\n\n"
        f"Peers' Round 2 positions:\n{peers}\n"
        f"Address {' and '.join(peer_names)} by name. Final AGREE/DISAGREE with visual evidence.\n\n"
        + _FMT_R3
    )

print("✅ Prompt templates defined")
print(f"   R1 prompt: {len(ROUND1_PROMPT)} chars")

## Cell 6: Response Parser (3-Pass Strategy)

In [ ]:
def strip_md(text):
    text = re.sub(r'```[\w]*\n?','',text)
    text = re.sub(r'\*{1,2}([^*]+)\*{1,2}',r'\1',text)
    return text.strip()

def _match_crop(t):
    """Pass 1 & 2: exact + substring match, longest wins."""
    t = t.lower()
    # Pass 1: exact
    for c in CDDM_CROPS:
        if c.lower() == t: return c
    # Pass 2: substring (longest match wins)
    matches = [c for c in CDDM_CROPS if c.lower() in t or t in c.lower()]
    return max(matches, key=len) if matches else None

def _match_disease(t):
    """Pass 1 & 2: exact + substring, longest wins."""
    t = t.lower()
    for d in CDDM_DISEASES:
        if d.lower() == t: return d
    matches = [d for d in CDDM_DISEASES if d.lower() in t or t in d.lower()]
    return max(matches, key=len) if matches else None

def _word_overlap(text_lower, taxonomy_list):
    """Pass 3: word-overlap scoring. Returns best match."""
    words = set(re.findall(r'\w+', text_lower))
    best, best_score = None, 0
    for item in taxonomy_list:
        item_words = set(re.findall(r'\w+', item.lower()))
        score = len(words & item_words) / max(len(item_words), 1)
        if score > best_score:
            best, best_score = item, score
    return best if best_score > 0 else None

def parse_response(text, agent_name=''):
    result = {
        'crop_category':    None,
        'disease_category': None,
        'diseased':         None,
        'reasoning':        '',
        'confidence':       None,
        'influenced_by':    None,
        'new_visual_evidence': None,
        'raw':              text,
    }
    if not text or text.startswith('ERROR'):
        return result

    clean   = strip_md(text)
    clean_l = clean.lower()
    lines   = clean.split('\n')
    in_rsn  = False; rsn_lines = []
    known   = ['crop category:','disease category:','diseased:',
               'reasoning:','confidence:','influenced by:','new visual evidence:']

    for line in lines:
        s = line.strip(); sl = s.lower()
        if any(sl.startswith(f) for f in known): in_rsn = False

        if sl.startswith('crop category:'):
            val = s.split(':',1)[1].strip()
            result['crop_category'] = _match_crop(val) or _word_overlap(val.lower(), CDDM_CROPS)
        elif sl.startswith('disease category:'):
            val = s.split(':',1)[1].strip()
            result['disease_category'] = _match_disease(val) or _word_overlap(val.lower(), CDDM_DISEASES)
        elif sl.startswith('diseased:'):
            val = s.split(':',1)[1].strip().lower()
            if re.search(r'\byes\b',val): result['diseased'] = 1
            elif re.search(r'\bno\b', val): result['diseased'] = 0
        elif sl.startswith('reasoning:'):
            result['reasoning'] = s.split(':',1)[1].strip(); in_rsn = True
        elif in_rsn and s and not any(sl.startswith(f) for f in known):
            rsn_lines.append(s)
        elif sl.startswith('confidence:'):
            nums = re.findall(r'[0-9]*\.?[0-9]+', s.split(':',1)[1])
            if nums:
                c = float(nums[0])
                result['confidence'] = c/100 if c>1 else c
        elif sl.startswith('influenced by:'):
            val = s.split(':',1)[1].strip()
            result['influenced_by'] = ('None' if val.lower() in
                ('none','n/a','na','no one','nobody','') else val)
        elif sl.startswith('new visual evidence:'):
            val = s.split(':',1)[1].strip()
            result['new_visual_evidence'] = val if val.upper() != 'NONE' else 'NONE'

    if rsn_lines:
        result['reasoning'] = (result['reasoning']+' '+' '.join(rsn_lines)).strip()

    # Fallback pass 3: word overlap on full response
    if result['crop_category'] is None:
        result['crop_category'] = _word_overlap(clean_l, CDDM_CROPS)
        if result['crop_category'] and agent_name:
            print(f"   ℹ️  [{agent_name}] crop fallback → {result['crop_category']}")
    if result['disease_category'] is None:
        result['disease_category'] = _word_overlap(clean_l, CDDM_DISEASES)
        if result['disease_category'] and agent_name:
            print(f"   ℹ️  [{agent_name}] disease fallback → {result['disease_category']}")
    if result['diseased'] is None and result['disease_category']:
        result['diseased'] = 0 if result['disease_category'] == HEALTHY_LABEL else 1
    if result['crop_category'] is None and agent_name:
        print(f"   ⚠️  [{agent_name}] CROP parse FAILED — {repr(text[:80])}")
    if result['disease_category'] is None and agent_name:
        print(f"   ⚠️  [{agent_name}] DISEASE parse FAILED — {repr(text[:80])}")
    return result

print("✅ 3-pass parser defined")

## Cell 7: Agent Inference Functions (All 6 Providers)

In [ ]:
def encode_b64(path):
    with open(path,'rb') as f: return base64.b64encode(f.read()).decode()

def mime(path):
    return 'image/jpeg' if str(path).lower().endswith(('jpg','jpeg')) else 'image/png'

# ── OpenAI (GPT-4.1, GPT-4.1-mini) ─────────────────────────────────────────────
def call_openai(model, image_path, prompt, retries=3, delay=5):
    b64 = encode_b64(image_path)
    for attempt in range(retries):
        try:
            resp = openai_client.chat.completions.create(
                model=model,
                messages=[{'role':'user','content':[
                    {'type':'image_url','image_url':{'url':f'data:{mime(image_path)};base64,{b64}','detail':'high'}},
                    {'type':'text','text':prompt}
                ]}],
                max_completion_tokens=700,
            )
            finish = resp.choices[0].finish_reason
            if finish != 'stop':
                print(f"   ⚠️  [{model}] finish={finish}")
            return (resp.choices[0].message.content or '').strip()
        except Exception as e:
            if attempt < retries-1:
                print(f"   ⚠️  OpenAI error ({e}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {e}"

# ── xAI Grok (OpenAI-compatible endpoint) ───────────────────────────────────────
def call_grok(model, image_path, prompt, retries=3, delay=5):
    b64 = encode_b64(image_path)
    for attempt in range(retries):
        try:
            resp = grok_client.chat.completions.create(
                model=model,
                messages=[{'role':'user','content':[
                    {'type':'image_url','image_url':{'url':f'data:{mime(image_path)};base64,{b64}'}},
                    {'type':'text','text':prompt}
                ]}],
                max_tokens=700,
                temperature=0.2,
            )
            return (resp.choices[0].message.content or '').strip()
        except Exception as e:
            if attempt < retries-1:
                print(f"   ⚠️  Grok error ({e}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {e}"

# ── Anthropic (Claude Sonnet 4.5, Claude Haiku 4.5) ─────────────────────────────
def call_anthropic(model, image_path, prompt, retries=3, delay=5):
    b64 = encode_b64(image_path)
    mt  = mime(image_path)
    for attempt in range(retries):
        try:
            resp = anthropic_client.messages.create(
                model=model,
                max_tokens=700,
                system="You are a plant science researcher. Always begin your response with the structured fields requested. Never add preamble.",
                messages=[{'role':'user','content':[
                    {'type':'image','source':{'type':'base64','media_type':mt,'data':b64}},
                    {'type':'text','text':prompt}
                ]}]
            )
            return resp.content[0].text.strip()
        except Exception as e:
            if attempt < retries-1:
                print(f"   ⚠️  Anthropic error ({e}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {e}"

# ── Together.ai Gemma (base64 via catbox.moe public URL) ────────────────────────
# Note: Together.ai serverless Gemma-3n-E4B does not accept inline base64.
# Images must be uploaded to a public URL first.
# Utility: upload_to_catbox(image_path) → public URL (requires internet access)
def upload_to_catbox(image_path):
    """Upload image to catbox.moe and return public URL."""
    import urllib.request, urllib.parse
    with open(image_path, 'rb') as f:
        data = f.read()
    boundary = 'boundary123456'
    body = (
        f'--{boundary}\r\nContent-Disposition: form-data; name="reqtype"\r\n\r\nfileupload\r\n'
        f'--{boundary}\r\nContent-Disposition: form-data; name="userhash"\r\n\r\n\r\n'
        f'--{boundary}\r\nContent-Disposition: form-data; name="fileToUpload"; filename="{Path(image_path).name}"\r\n'
        f'Content-Type: {mime(image_path)}\r\n\r\n'
    ).encode() + data + f'\r\n--{boundary}--\r\n'.encode()
    req = urllib.request.Request(
        'https://catbox.moe/user/api.php',
        data=body,
        headers={'Content-Type': f'multipart/form-data; boundary={boundary}'}
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return resp.read().decode().strip()

def call_together_gemma(model, image_path, prompt, retries=3, delay=8):
    for attempt in range(retries):
        try:
            url = upload_to_catbox(image_path)
            resp = together_client.chat.completions.create(
                model=model,
                messages=[{'role':'user','content':[
                    {'type':'text','text':prompt},
                    {'type':'image_url','image_url':{'url': url}}
                ]}],
                max_tokens=700,
                temperature=0.2,
            )
            return (resp.choices[0].message.content or '').strip()
        except Exception as e:
            if attempt < retries-1:
                print(f"   ⚠️  Together/Gemma error ({e}), retry {attempt+1}...")
                time.sleep(delay)
            else: return f"ERROR: {e}"

# ── Agent Registry ────────────────────────────────────────────────────────────────
AGENTS_VIDA = {
    'GPT-4.1':        lambda img, p: call_openai('gpt-4.1',        img, p),
    'Grok-4.20-Fast': lambda img, p: call_grok('grok-4.20-non-reasoning', img, p),
    'GPT-4.1-mini':   lambda img, p: call_openai('gpt-4.1-mini',   img, p),
    'Claude-Sonnet':  lambda img, p: call_anthropic('claude-sonnet-4-5-20251001', img, p),
    'Claude-Haiku':   lambda img, p: call_anthropic('claude-haiku-4-5-20251001',  img, p),
    'Gemma-3n-E4B':   lambda img, p: call_together_gemma('google/gemma-3n-E4B-it', img, p),
}
print(f"✅ {len(AGENTS_VIDA)} agents registered: {list(AGENTS_VIDA.keys())}")

## Cell 8: Softmax-Weighted Voting (T=2.0)

In [ ]:
# ── Softmax weights from R1 Combined Accuracy ──────────────────────────────────
# w_i = exp(acc_i / T) / sum(exp(acc_j / T)),  T = 2.0
# T=2.0 gives ~2:1 weight ratio for a 0.8 vs 0.5 accuracy gap (softened, not winner-take-all)
# Weights computed once after VIDA, applied to both pre-debate baseline and post-debate R3 votes.

def compute_softmax_weights(agent_names, combined_accs):
    """agent_names: list, combined_accs: dict {name: float}"""
    accs = np.array([combined_accs.get(a, 0.0) for a in agent_names])
    exp_a   = np.exp(accs / SOFTMAX_TEMP)
    weights = exp_a / exp_a.sum()
    w_dict  = dict(zip(agent_names, weights))
    print(f"   📊 Softmax weights (T={SOFTMAX_TEMP}):")
    for name, w in sorted(w_dict.items(), key=lambda x: -x[1]):
        acc = combined_accs.get(name, 0.0)
        print(f"      {name:<22}: acc={acc:.3f} → weight={w:.3f}")
    return w_dict

def weighted_vote(values_dict, weights_dict):
    """values_dict: {agent: label}. Returns (winner, weighted_scores, agreement)."""
    scores = {}
    for agent, val in values_dict.items():
        if val is None: continue
        w = weights_dict.get(agent, 1.0 / max(len(values_dict), 1))
        scores[val] = scores.get(val, 0.0) + w
    if not scores: return None, {}, 0.0
    winner = max(scores, key=scores.get)
    return winner, scores, scores[winner]

def aggregate_votes(parsed_dict, weights_dict, round_label='r3'):
    """Aggregate weighted votes for one round. Returns dict of consensus fields."""
    agents    = list(parsed_dict.keys())
    crop_vals = {a: parsed_dict[a]['crop_category']    for a in agents}
    dis_vals  = {a: parsed_dict[a]['disease_category'] for a in agents}
    dis_bin   = {a: parsed_dict[a]['diseased']         for a in agents}

    crop_w, _, crop_ag = weighted_vote(crop_vals, weights_dict)
    dis_w,  _, dis_ag  = weighted_vote(dis_vals,  weights_dict)
    dis_bw, _, _        = weighted_vote(dis_bin,   weights_dict)

    confs = [parsed_dict[a]['confidence'] for a in agents if parsed_dict[a]['confidence'] is not None]
    maj   = [a for a in agents if parsed_dict[a]['disease_category'] == dis_w]
    rsn   = ' | '.join(parsed_dict[a]['reasoning'] for a in maj if parsed_dict[a]['reasoning'])
    pfx   = f'{round_label}_'
    return {
        f'{pfx}consensus_crop':      crop_w,
        f'{pfx}consensus_disease':   dis_w,
        f'{pfx}consensus_diseased':  dis_bw,
        f'{pfx}consensus_reasoning': rsn,
        f'{pfx}mean_confidence':     float(np.mean(confs)) if confs else None,
        f'{pfx}crop_agreement':      round(crop_ag, 3),
        f'{pfx}disease_agreement':   round(dis_ag, 3),
        f'{pfx}individual_crops':    {a: parsed_dict[a]['crop_category']    for a in agents},
        f'{pfx}individual_diseases': {a: parsed_dict[a]['disease_category'] for a in agents},
    }

print("✅ Softmax-weighted voting defined")

## Cell 9: GPT-4.1 Judge — Reasoning Quality & Plausibility

In [ ]:
def judge_response(gt_crop, gt_disease, agent_reasoning, pred_crop, pred_disease):
    """GPT-4.1 scores Reasoning Quality and Plausibility (1-10 each)."""
    if not agent_reasoning:
        return {'reasoning_quality': None, 'plausibility': None}
    judge_prompt = (
        f"Ground truth: {gt_crop} / {gt_disease}\n"
        f"Agent predicted: {pred_crop} / {pred_disease}\n"
        f"Agent reasoning: {agent_reasoning}\n\n"
        "Rate this agricultural diagnosis reasoning on two dimensions (1-10 each):\n"
        "Reasoning Quality: <1-10 — does the reasoning cite specific visual features "
        "(leaf shape, lesion color, spot pattern, texture, margins, etc.)?>\n"
        "Plausibility: <1-10 — even if wrong, is the reasoning scientifically plausible "
        "for a plant disease diagnosis?>\n"
        "Respond ONLY with those two lines, no other text."
    )
    try:
        resp = openai_client.chat.completions.create(
            model='gpt-4.1',
            messages=[{'role':'user','content':judge_prompt}],
            max_completion_tokens=60,
        )
        text = resp.choices[0].message.content or ''
        rq = re.search(r'Reasoning Quality:\s*([0-9.]+)', text)
        pl = re.search(r'Plausibility:\s*([0-9.]+)', text)
        return {
            'reasoning_quality': float(rq.group(1)) if rq else None,
            'plausibility':      float(pl.group(1)) if pl else None,
        }
    except Exception as e:
        return {'reasoning_quality': None, 'plausibility': None}

print("✅ GPT-4.1 judge defined")

## Cell 10: VIDA — Run Round 1 (All 6 Agents Independent)

In [ ]:
# ── VIDA: Round 1 independent analysis ─────────────────────────────────────────
vida_results  = []   # one row per (image, agent)
vida_raw_bank = {}   # {image_name: {agent_name: raw_response}}

agent_names = list(AGENTS_VIDA.keys())
print(f"🚀 VIDA: {len(pilot)} images × {len(agent_names)} agents (Round 1 independent)")
print(f"   Estimated: ~{len(pilot)*len(agent_names)*8//60}–{len(pilot)*len(agent_names)*12//60} min\n")

for i, sample in enumerate(tqdm(pilot, desc="VIDA")):
    img_path    = sample['image_path']
    img_name    = img_path.name
    gt_crop     = sample['gt_crop']
    gt_disease  = sample['gt_disease']
    gt_diseased = sample['gt_diseased']

    print(f"\n[{i+1}/{len(pilot)}] {img_name} | GT: {gt_crop} / {gt_disease}")
    vida_raw_bank[img_name] = {}

    for agent_name, fn in AGENTS_VIDA.items():
        raw    = fn(img_path, ROUND1_PROMPT)
        parsed = parse_response(raw, agent_name=f'{agent_name} R1')
        vida_raw_bank[img_name][agent_name] = raw

        crop_correct = int(parsed['crop_category']    == gt_crop)    if parsed['crop_category']    else 0
        dis_correct  = int(parsed['disease_category'] == gt_disease) if parsed['disease_category'] else 0
        parse_failed = int(parsed['crop_category'] is None or parsed['disease_category'] is None)

        vida_results.append({
            'image_name':      img_name,
            'class_label':     sample['class_label'],
            'gt_crop':         gt_crop,
            'gt_disease':      gt_disease,
            'gt_diseased':     gt_diseased,
            'agent':           agent_name,
            'pred_crop':       parsed['crop_category'],
            'pred_disease':    parsed['disease_category'],
            'pred_diseased':   parsed['diseased'],
            'reasoning':       parsed['reasoning'],
            'confidence':      parsed['confidence'],
            'crop_correct':    crop_correct,
            'disease_correct': dis_correct,
            'both_correct':    int(crop_correct and dis_correct),
            'parse_failed':    parse_failed,
        })
        status = '✅' if crop_correct and dis_correct else ('🌿' if crop_correct else '❌')
        print(f"   {status} {agent_name}: {parsed['crop_category']} / {parsed['disease_category']} (conf={parsed['confidence']})")

    # Checkpoint
    if (i+1) % CHECKPOINT == 0:
        pd.DataFrame(vida_results).to_csv(OUTPUT_DIR / 'vida_checkpoint.csv', index=False)
        with open(OUTPUT_DIR / 'vida_raw_bank.json','w') as f:
            json.dump(vida_raw_bank, f, indent=2)
        print(f"   💾 Checkpoint saved ({i+1}/{len(pilot)})")

# Save
vida_df = pd.DataFrame(vida_results)
vida_df.to_csv(OUTPUT_DIR / 'vida_r1_results.csv', index=False)
with open(OUTPUT_DIR / 'vida_raw_bank.json','w') as f:
    json.dump(vida_raw_bank, f, indent=2)
print(f"\n✅ VIDA complete. {len(vida_results)} rows saved.")

## Cell 11: GPT-4.1 Judge Scoring (Reasoning Quality + Plausibility)

In [ ]:
print(f"⏳ Running GPT-4.1 judge on {len(vida_results)} R1 responses...")

judge_scores = []
for row in tqdm(vida_results, desc="Judging"):
    scores = judge_response(
        row['gt_crop'], row['gt_disease'],
        row['reasoning'], row['pred_crop'], row['pred_disease']
    )
    judge_scores.append(scores)

vida_df['reasoning_quality'] = [s['reasoning_quality'] for s in judge_scores]
vida_df['plausibility']      = [s['plausibility']      for s in judge_scores]
vida_df.to_csv(OUTPUT_DIR / 'vida_r1_results.csv', index=False)
print(f"\n✅ Judge scores added and saved.")

## Cell 12: VIDA Metrics Dashboard

In [ ]:
def calibration_r(group):
    sub = group.dropna(subset=['confidence'])
    if len(sub) < 3: return None
    corr = sub['confidence'].corr(sub['both_correct'].astype(float))
    return round(corr, 3) if not np.isnan(corr) else None

metrics_rows = []
for agent in agent_names:
    g = vida_df[vida_df['agent'] == agent]
    metrics_rows.append({
        'Agent':             agent,
        'N':                 len(g),
        'Crop Acc':          round(g['crop_correct'].mean(), 3),
        'Disease Acc':       round(g['disease_correct'].mean(), 3),
        'Combined Acc':      round(g['both_correct'].mean(), 3),
        'Parse Fail %':      round(g['parse_failed'].mean() * 100, 1),
        'Mean Confidence':   round(g['confidence'].dropna().mean(), 3) if g['confidence'].notna().any() else None,
        'Conf Calibration':  calibration_r(g),
        'Reasoning Quality': round(g['reasoning_quality'].dropna().mean(), 2) if g['reasoning_quality'].notna().any() else None,
        'Plausibility':      round(g['plausibility'].dropna().mean(), 2)      if g['plausibility'].notna().any()      else None,
    })

metrics_df = pd.DataFrame(metrics_rows).set_index('Agent')
metrics_df = metrics_df.sort_values('Combined Acc', ascending=False)
metrics_df.to_csv(OUTPUT_DIR / 'vida_metrics.csv')

print("📊 VIDA Metrics (sorted by Combined Accuracy):")
print(metrics_df.to_string())

# ── 9-panel dashboard ──────────────────────────────────────────────────────────
agents_ordered = metrics_df.index.tolist()
fig = plt.figure(figsize=(24, 20))
fig.suptitle('VIDA Block — Multi-Agent Tournament Dashboard (CDDM, N=490)',
             fontsize=14, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)
colors = plt.cm.tab10(np.linspace(0, 1, len(agents_ordered)))

def bar_chart(ax, values, title, ylabel, color_map=None, ylim=(0,1.05)):
    x = np.arange(len(agents_ordered))
    bars = ax.bar(x, values, color=color_map if color_map else colors)
    ax.set_xticks(x)
    ax.set_xticklabels([a.replace('-',' ') for a in agents_ordered], rotation=30, ha='right', fontsize=8)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=9)
    if ylim: ax.set_ylim(ylim)
    ax.grid(axis='y', alpha=0.3)
    for bar, v in zip(bars, values):
        if v is not None and not np.isnan(float(v if v else 0)):
            ax.text(bar.get_x()+bar.get_width()/2, float(v)+0.01, f'{float(v):.2f}',
                    ha='center', fontsize=7, fontweight='bold')

ax = fig.add_subplot(gs[0,0])
bar_chart(ax, [metrics_df.loc[a,'Crop Acc'] for a in agents_ordered], 'Crop Accuracy', 'Accuracy')
ax = fig.add_subplot(gs[0,1])
bar_chart(ax, [metrics_df.loc[a,'Disease Acc'] for a in agents_ordered], 'Disease Accuracy', 'Accuracy')
ax = fig.add_subplot(gs[0,2])
bar_chart(ax, [metrics_df.loc[a,'Combined Acc'] for a in agents_ordered], 'Combined Accuracy', 'Accuracy',
          color_map=['#2a9d8f' if i==0 else '#e76f51' if i==len(agents_ordered)-1 else '#8ECAE6'
                     for i in range(len(agents_ordered))])
ax = fig.add_subplot(gs[1,0])
bar_chart(ax, [metrics_df.loc[a,'Parse Fail %'] for a in agents_ordered], 'Parse Failure % (lower=better)', '%',
          color_map=['#e63946' if v>20 else '#f4a261' if v>5 else '#2a9d8f'
                     for v in [metrics_df.loc[a,'Parse Fail %'] for a in agents_ordered]],
          ylim=(0, max(metrics_df['Parse Fail %'].fillna(0).max()*1.2, 10)))
ax = fig.add_subplot(gs[1,1])
conf_vals = [metrics_df.loc[a,'Mean Confidence'] if pd.notna(metrics_df.loc[a,'Mean Confidence']) else 0
             for a in agents_ordered]
bar_chart(ax, conf_vals, 'Mean Confidence', 'Confidence')
ax = fig.add_subplot(gs[1,2])
cal_vals = [metrics_df.loc[a,'Conf Calibration'] if pd.notna(metrics_df.loc[a,'Conf Calibration']) else 0
            for a in agents_ordered]
bar_chart(ax, cal_vals, 'Confidence Calibration (Pearson r)', 'r', ylim=(-1.1,1.1))
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax = fig.add_subplot(gs[2,0])
rq_vals = [metrics_df.loc[a,'Reasoning Quality'] if pd.notna(metrics_df.loc[a,'Reasoning Quality']) else 0
           for a in agents_ordered]
bar_chart(ax, rq_vals, 'Reasoning Quality (GPT-4.1 Judge)', 'Score /10', ylim=(0,10.5))
ax = fig.add_subplot(gs[2,1])
pl_vals = [metrics_df.loc[a,'Plausibility'] if pd.notna(metrics_df.loc[a,'Plausibility']) else 0
           for a in agents_ordered]
bar_chart(ax, pl_vals, 'Plausibility (GPT-4.1 Judge)', 'Score /10', ylim=(0,10.5))
ax = fig.add_subplot(gs[2,2])
heat_cols = ['Crop Acc','Disease Acc','Combined Acc','Reasoning Quality','Plausibility']
heat_data = metrics_df[heat_cols].copy()
for col in heat_cols:
    mn, mx = heat_data[col].min(), heat_data[col].max()
    heat_data[col] = (heat_data[col] - mn) / (mx - mn + 1e-9)
sns.heatmap(heat_data, annot=metrics_df[heat_cols].round(2), fmt='', cmap='RdYlGn', ax=ax,
            xticklabels=[c.replace(' ','
') for c in heat_cols], yticklabels=agents_ordered)
ax.set_title('Normalized Performance Heatmap', fontsize=10, fontweight='bold')
ax.tick_params(labelsize=7)

plt.savefig(OUTPUT_DIR / 'vida_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved to vida_dashboard.png")

## Cell 13: ⚙️ PANDA Agent Selection

Review the VIDA dashboard, then configure below.

- `AUTO_SELECT = True` → picks top-`N_DEBATE` agents automatically by Combined Accuracy  
- `AUTO_SELECT = False` → use `MANUAL_AGENTS` list

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit before running PANDA
# ════════════════════════════════════════════════════════════════════════════════

AUTO_SELECT   = True   # True = auto top-N_DEBATE | False = use MANUAL_AGENTS

MANUAL_AGENTS = [      # Only used when AUTO_SELECT = False
    "GPT-4.1",
    "GPT-4.1-mini",
    "Grok-4.20-Fast",
]

# ════════════════════════════════════════════════════════════════════════════════

if AUTO_SELECT:
    sorted_agents = metrics_df.sort_values('Combined Acc', ascending=False).index.tolist()
    DEBATE_AGENTS = sorted_agents[:N_DEBATE]
    print(f"✅ AUTO-SELECTED top-{N_DEBATE} by Combined Accuracy: {DEBATE_AGENTS}")
else:
    invalid = [a for a in MANUAL_AGENTS if a not in agent_names]
    if invalid:
        print(f"❌ Invalid agent names: {invalid}")
        print(f"   Valid names: {agent_names}")
    else:
        DEBATE_AGENTS = MANUAL_AGENTS
        print(f"✅ MANUAL selection: {DEBATE_AGENTS}")

print(f"\n📊 Their VIDA stats:")
print(metrics_df.loc[DEBATE_AGENTS, ['Crop Acc','Disease Acc','Combined Acc','Reasoning Quality']].to_string())

# Compute softmax voting weights
combined_accs_dict = metrics_df['Combined Acc'].to_dict()
AGENT_WEIGHTS = compute_softmax_weights(DEBATE_AGENTS, combined_accs_dict)

# Build debate agent fn map
AGENT_FN_MAP = {
    'GPT-4.1':        lambda img, p: call_openai('gpt-4.1',        img, p),
    'Grok-4.20-Fast': lambda img, p: call_grok('grok-4.20-non-reasoning', img, p),
    'GPT-4.1-mini':   lambda img, p: call_openai('gpt-4.1-mini',   img, p),
    'Claude-Sonnet':  lambda img, p: call_anthropic('claude-sonnet-4-5-20251001', img, p),
    'Claude-Haiku':   lambda img, p: call_anthropic('claude-haiku-4-5-20251001',  img, p),
    'Gemma-3n-E4B':   lambda img, p: call_together_gemma('google/gemma-3n-E4B-it', img, p),
}
AGENTS_PANDA = {name: AGENT_FN_MAP[name] for name in DEBATE_AGENTS}
print(f"\n✅ PANDA agents ready: {list(AGENTS_PANDA.keys())}")

## Cell 14: PANDA — Run Rounds 2 + 3 (Named Peer Deliberation)

In [ ]:
panda_results    = []
panda_transcripts= {}
debate_agents    = list(AGENTS_PANDA.keys())

print(f"🚀 PANDA Debate: {len(pilot)} images × {len(debate_agents)} agents × 2 rounds (R2 + R3)")
print(f"   Debate agents: {debate_agents}\n")

for i, sample in enumerate(tqdm(pilot, desc="PANDA")):
    img_path    = sample['image_path']
    img_name    = img_path.name
    gt_crop     = sample['gt_crop']
    gt_disease  = sample['gt_disease']
    gt_diseased = sample['gt_diseased']

    print(f"\n[{i+1}/{len(pilot)}] {img_name} | GT: {gt_crop} / {gt_disease}")

    # ── Load R1 from VIDA bank ────────────────────────────────────────────────
    r1_raw_from_bank = vida_raw_bank.get(img_name, {})
    r1_parsed = {name: parse_response(r1_raw_from_bank.get(name,''), agent_name=f'{name} R1-load')
                 for name in debate_agents}

    result = {
        'image_name': img_name, 'class_label': sample['class_label'],
        'gt_crop': gt_crop, 'gt_disease': gt_disease, 'gt_diseased': gt_diseased,
    }
    for name in debate_agents:
        result[f'r1_{name}_crop']    = r1_parsed[name]['crop_category']
        result[f'r1_{name}_disease'] = r1_parsed[name]['disease_category']

    # ── Round 2: Named peer debate ────────────────────────────────────────────
    print("  [R2] Named peer debate...")
    r2_raw = {}; r2_parsed = {}
    for name, fn in AGENTS_PANDA.items():
        others_r1 = {k: v for k,v in r1_parsed.items() if k != name}
        raw = fn(img_path, make_r2_prompt(name, r1_parsed[name], others_r1))
        r2_raw[name]    = raw
        r2_parsed[name] = parse_response(raw, agent_name=f'{name} R2')
        result[f'r2_{name}_crop']               = r2_parsed[name]['crop_category']
        result[f'r2_{name}_disease']            = r2_parsed[name]['disease_category']
        result[f'r2_{name}_influenced_by']      = r2_parsed[name]['influenced_by']
        result[f'r2_{name}_new_visual_evidence']= r2_parsed[name]['new_visual_evidence']
        r1_dis = r1_parsed[name]['disease_category']
        r2_dis = r2_parsed[name]['disease_category']
        result[f'r2_{name}_disease_changed'] = int(r1_dis and r2_dis and r1_dis != r2_dis)
        r1_crop = r1_parsed[name]['crop_category']
        r2_crop = r2_parsed[name]['crop_category']
        result[f'r2_{name}_crop_changed'] = int(r1_crop and r2_crop and r1_crop != r2_crop)
        print(f"    {name}: {r2_parsed[name]['crop_category']} / {r2_parsed[name]['disease_category']}"
              f" | inf={r2_parsed[name]['influenced_by']}")

    # ── Round 3: Final verdict ────────────────────────────────────────────────
    print("  [R3] Final verdict...")
    r3_raw = {}; r3_parsed = {}
    for name, fn in AGENTS_PANDA.items():
        others_r2 = {k: v for k,v in r2_parsed.items() if k != name}
        raw = fn(img_path, make_r3_prompt(name, r1_parsed[name], r2_parsed[name], others_r2))
        r3_raw[name]    = raw
        r3_parsed[name] = parse_response(raw, agent_name=f'{name} R3')
        result[f'r3_{name}_crop']               = r3_parsed[name]['crop_category']
        result[f'r3_{name}_disease']            = r3_parsed[name]['disease_category']
        result[f'r3_{name}_confidence']         = r3_parsed[name]['confidence']
        result[f'r3_{name}_influenced_by']      = r3_parsed[name]['influenced_by']
        result[f'r3_{name}_new_visual_evidence']= r3_parsed[name]['new_visual_evidence']
        r2_dis = r2_parsed[name]['disease_category']
        r3_dis = r3_parsed[name]['disease_category']
        result[f'r3_{name}_disease_changed'] = int(r2_dis and r3_dis and r2_dis != r3_dis)
        r2_crop = r2_parsed[name]['crop_category']
        r3_crop = r3_parsed[name]['crop_category']
        result[f'r3_{name}_crop_changed'] = int(r2_crop and r3_crop and r2_crop != r3_crop)
        print(f"    {name}: {r3_parsed[name]['crop_category']} / {r3_parsed[name]['disease_category']}"
              f" conf={r3_parsed[name]['confidence']} inf={r3_parsed[name]['influenced_by']}")

    # ── Weighted consensus ────────────────────────────────────────────────────
    post = aggregate_votes(r3_parsed, AGENT_WEIGHTS, round_label='r3')
    pre  = aggregate_votes(r1_parsed, AGENT_WEIGHTS, round_label='r1_baseline')
    result.update(post); result.update(pre)

    # Correctness — post-debate (R3)
    result['consensus_crop_correct']    = int(post['r3_consensus_crop']    == gt_crop)    if post['r3_consensus_crop']    else 0
    result['consensus_disease_correct'] = int(post['r3_consensus_disease'] == gt_disease) if post['r3_consensus_disease'] else 0
    result['consensus_both_correct']    = int(result['consensus_crop_correct'] and result['consensus_disease_correct'])
    # Correctness — pre-debate baseline (R1 weighted)
    result['baseline_crop_correct']    = int(pre.get('r1_baseline_consensus_crop','')    == gt_crop)
    result['baseline_disease_correct'] = int(pre.get('r1_baseline_consensus_disease','') == gt_disease)
    result['baseline_both_correct']    = int(result['baseline_crop_correct'] and result['baseline_disease_correct'])

    # Per-agent correctness R1→R3
    for name in debate_agents:
        for rnd, parsed_d in [('r1',r1_parsed),('r2',r2_parsed),('r3',r3_parsed)]:
            p = parsed_d[name]
            result[f'{rnd}_{name}_crop_correct']    = int(p['crop_category']    == gt_crop)    if p['crop_category']    else 0
            result[f'{rnd}_{name}_disease_correct'] = int(p['disease_category'] == gt_disease) if p['disease_category'] else 0
            result[f'{rnd}_{name}_both_correct']    = int(result[f'{rnd}_{name}_crop_correct'] and result[f'{rnd}_{name}_disease_correct'])

    panda_results.append(result)
    panda_transcripts[img_name] = {'r1_raw': r1_raw_from_bank, 'r2': r2_raw, 'r3': r3_raw}

    # Checkpoint
    if (i+1) % CHECKPOINT == 0:
        pd.DataFrame(panda_results).to_csv(OUTPUT_DIR / 'panda_checkpoint.csv', index=False)
        print(f"   💾 Checkpoint ({i+1}/{len(pilot)})")

panda_df = pd.DataFrame(panda_results)
panda_df.to_csv(OUTPUT_DIR / 'panda_results.csv', index=False)
with open(OUTPUT_DIR / 'panda_transcripts.json','w') as f:
    json.dump(panda_transcripts, f, indent=2)
print(f"\n✅ PANDA complete. {len(panda_results)} rows saved.")

## Cell 15: Influence Analysis, McNemar, & Visualization

In [ ]:
# ── Pre/post debate comparison ──────────────────────────────────────────────────
print("="*70)
print("  PANDA RESULTS: Pre-Debate Baseline vs Post-Debate Consensus")
print("="*70)
if 'baseline_both_correct' in panda_df.columns:
    base_crop = panda_df['baseline_crop_correct'].mean()
    base_dis  = panda_df['baseline_disease_correct'].mean()
    base_both = panda_df['baseline_both_correct'].mean()
    cons_crop = panda_df['consensus_crop_correct'].mean()
    cons_dis  = panda_df['consensus_disease_correct'].mean()
    cons_both = panda_df['consensus_both_correct'].mean()
    delta = cons_both - base_both
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '→')
    print(f"  {'':28} {'Crop':>8} {'Disease':>9} {'Combined':>10}")
    print("  " + "-"*58)
    print(f"  {'Pre-Debate (R1 weighted):':<28} {base_crop:>8.3f} {base_dis:>9.3f} {base_both:>10.3f}")
    print(f"  {'Post-Debate (R3 weighted):':<28} {cons_crop:>8.3f} {cons_dis:>9.3f} {cons_both:>10.3f}  {arrow} Δ={delta:+.3f}")
    print()
    if delta > 0.01:
        print(f"  ✅ Debate IMPROVED combined accuracy by {delta:.1%}")
    elif delta < -0.01:
        print(f"  ❌ Debate HURT combined accuracy by {abs(delta):.1%}")
    else:
        print(f"  → Debate had no significant effect on combined accuracy (consistent with prior work)")

print()
print("="*70)
print("  PER-AGENT: R1 Independent → R3 Post-Debate")
print("="*70)
print(f"  {'Agent':<22} {'R1 Crop':>8} {'R1 Dis':>8} {'R1 Both':>8} │ {'R3 Crop':>8} {'R3 Dis':>8} {'R3 Both':>8}  Weight")
print("  " + "-"*80)
for name in debate_agents:
    r1c = panda_df.apply(lambda r: r.get(f'r1_{name}_crop_correct',0), axis=1).mean()
    r1d = panda_df.apply(lambda r: r.get(f'r1_{name}_disease_correct',0), axis=1).mean()
    r1b = panda_df.apply(lambda r: r.get(f'r1_{name}_both_correct',0), axis=1).mean()
    r3c = panda_df.apply(lambda r: r.get(f'r3_{name}_crop_correct',0), axis=1).mean()
    r3d = panda_df.apply(lambda r: r.get(f'r3_{name}_disease_correct',0), axis=1).mean()
    r3b = panda_df.apply(lambda r: r.get(f'r3_{name}_both_correct',0), axis=1).mean()
    delta_name = r3b - r1b
    arrow = '↑' if delta_name > 0.01 else ('↓' if delta_name < -0.01 else '→')
    wt = AGENT_WEIGHTS.get(name, 0)
    print(f"  {name:<22} {r1c:>8.3f} {r1d:>8.3f} {r1b:>8.3f} │ {r3c:>8.3f} {r3d:>8.3f} {r3b:>8.3f}  {arrow} w={wt:.3f}")

# ── Influence analysis ────────────────────────────────────────────────────────────
influence_matrix  = defaultdict(int)
stubbornness      = {n: 0 for n in debate_agents}
flip_flop         = {n: 0 for n in debate_agents}
change_helped_dis = {n: 0 for n in debate_agents}
change_hurt_dis   = {n: 0 for n in debate_agents}
n_img = len(panda_df)

for _, row in panda_df.iterrows():
    for name in debate_agents:
        ch_r2 = row.get(f'r2_{name}_disease_changed', 0)
        ch_r3 = row.get(f'r3_{name}_disease_changed', 0)
        if not ch_r2 and not ch_r3: stubbornness[name] += 1
        if ch_r2 and ch_r3:         flip_flop[name] += 1
        if ch_r2:
            r1_c = int(row.get(f'r1_{name}_disease_correct', 0))
            r2_c = int(row.get(f'r2_{name}_disease_correct', 0))
            if not r1_c and r2_c:   change_helped_dis[name] += 1
            elif r1_c and not r2_c: change_hurt_dis[name]   += 1
        for rnd in ['r2','r3']:
            inf_raw = row.get(f'{rnd}_{name}_influenced_by')
            if inf_raw and inf_raw != 'None' and pd.notna(inf_raw):
                for candidate in debate_agents:
                    if candidate.lower() in str(inf_raw).lower():
                        influence_matrix[(candidate, name)] += 1; break

print()
print("="*70)
print("  PEER INFLUENCE MATRIX (rows=persuader, cols=changed)")
print("="*70)
print(f"  {'Agent':<22}", end='')
for n in debate_agents: print(f" {n:>16}", end='')
print()
for inf_n in debate_agents:
    print(f"  {inf_n:<22}", end='')
    for chg_n in debate_agents:
        v = '—' if inf_n==chg_n else str(influence_matrix.get((inf_n,chg_n),0))
        print(f" {v:>16}", end='')
    print()

print()
print(f"  {'Agent':<22} {'Stubborn%':>10} {'Flip-flop%':>11} {'Helped':>7} {'Hurt':>6}")
for name in debate_agents:
    print(f"  {name:<22} {stubbornness[name]/n_img:>10.1%} {flip_flop[name]/n_img:>11.1%}"
          f" {change_helped_dis[name]:>7} {change_hurt_dis[name]:>6}")

# Save influence matrix
inf_rows = [{'influencer':k[0],'changed_agent':k[1],'count':v} for k,v in influence_matrix.items()]
if inf_rows: pd.DataFrame(inf_rows).to_csv(OUTPUT_DIR / 'influence_matrix.csv', index=False)

# ── Visualization ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle(f'PANDA Debate Analysis — {", ".join(debate_agents)}  (N={len(panda_df)})',
             fontsize=13, fontweight='bold', y=1.01)
x = np.arange(len(debate_agents)); w = 0.3
colors_blue = ['#8ECAE6','#219EBC','#023047']

def compute_acc(df, agents, rnd):
    crops = [df.apply(lambda r: int(r.get(f'{rnd}_{n}_crop_correct',0)==1), axis=1).mean() for n in agents]
    discs = [df.apply(lambda r: int(r.get(f'{rnd}_{n}_disease_correct',0)==1), axis=1).mean() for n in agents]
    boths = [df.apply(lambda r: int(r.get(f'{rnd}_{n}_both_correct',0)==1), axis=1).mean() for n in agents]
    return crops, discs, boths

r1_crop, r1_dis, r1_both = compute_acc(panda_df, debate_agents, 'r1')
r3_crop, r3_dis, r3_both = compute_acc(panda_df, debate_agents, 'r3')
cons_crop_acc = panda_df['consensus_crop_correct'].mean()
cons_dis_acc  = panda_df['consensus_disease_correct'].mean()
cons_both_acc = panda_df['consensus_both_correct'].mean()

for ax_idx, (vals_r1, vals_r3, cons_v, title) in enumerate([
    (r1_crop, r3_crop, cons_crop_acc, 'Crop Accuracy: R1 vs R3'),
    (r1_dis,  r3_dis,  cons_dis_acc,  'Disease Accuracy: R1 vs R3'),
    (r1_both, r3_both, cons_both_acc, 'Combined Accuracy: R1 vs R3'),
]):
    ax = axes[0, ax_idx]
    ax.bar(x-w/2, vals_r1, w, label='R1 (pre-debate)', color=colors_blue[0])
    ax.bar(x+w/2, vals_r3, w, label='R3 (post-debate)', color=colors_blue[1])
    ax.axhline(cons_v, color='red', ls='--', lw=1.5, label=f'Consensus ({cons_v:.2f})')
    ax.set_xticks(x); ax.set_xticklabels(debate_agents, rotation=15, fontsize=9)
    ax.set_ylim(0,1.05); ax.set_title(title, fontsize=10); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.4)

ax = axes[1,0]
base_both_acc = panda_df['baseline_both_correct'].mean() if 'baseline_both_correct' in panda_df else 0
ax.bar(['Pre-debate
(R1 weighted)', 'Post-debate
(R3 weighted)'],
       [base_both_acc, cons_both_acc], color=['#8ECAE6','#219EBC'], width=0.5)
ax.set_ylim(0, 1.0); ax.set_ylabel('Combined Accuracy')
ax.set_title(f'Debate Impact: Δ={cons_both_acc-base_both_acc:+.3f}', fontsize=10)
ax.grid(axis='y', alpha=0.4)

ax = axes[1,1]
inf_mat = np.zeros((len(debate_agents), len(debate_agents)))
for i2, inf_n in enumerate(debate_agents):
    for j2, chg_n in enumerate(debate_agents):
        if i2 != j2: inf_mat[i2,j2] = influence_matrix.get((inf_n,chg_n),0)
sns.heatmap(inf_mat, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            xticklabels=debate_agents, yticklabels=debate_agents,
            cbar_kws={'label':'Times influenced'})
ax.set_xlabel('Agent CHANGED'); ax.set_ylabel('PERSUADER')
ax.set_title('Peer Influence Matrix', fontsize=10); ax.tick_params(rotation=20, labelsize=8)

ax = axes[1,2]
stub_v = [stubbornness[n]/n_img for n in debate_agents]
flip_v = [flip_flop[n]/n_img    for n in debate_agents]
r2ch_v = [panda_df.apply(lambda r: r.get(f'r2_{n}_disease_changed',0), axis=1).mean() for n in debate_agents]
x4 = np.arange(len(debate_agents)); w4 = 0.22
ax.bar(x4-w4,   stub_v, w4, label='Stubborn',     color='#264653')
ax.bar(x4,      r2ch_v, w4, label='R1→R2 change', color=colors_blue[0])
ax.bar(x4+w4,   flip_v, w4, label='Flip-flop',    color='#e76f51')
ax.set_xticks(x4); ax.set_xticklabels(debate_agents, rotation=15, fontsize=9)
ax.set_ylim(0,1.05); ax.set_title('Agent Behavior', fontsize=10); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'panda_debate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ PANDA visualization saved to panda_debate_analysis.png")

## Cell 16: Full Summary Report

In [ ]:
print("="*70)
print("  VIDA + PANDA — FULL SUMMARY REPORT")
print("="*70)
print(f"  Dataset      : CDDM — {len(pilot)} images from {len(set(p['class_label'] for p in pilot))} classes")
print(f"  VIDA agents  : {len(agent_names)} ({', '.join(agent_names)})")
print(f"  PANDA agents : {len(debate_agents)} ({', '.join(debate_agents)})")
print(f"  Voting       : Softmax-weighted (T={SOFTMAX_TEMP})")
print()
print("  VIDA RANKINGS (by Combined Accuracy):")
for rank, agent in enumerate(metrics_df.index.tolist(), 1):
    ca = metrics_df.loc[agent,'Combined Acc']
    rq = metrics_df.loc[agent,'Reasoning Quality'] or 0
    wt = AGENT_WEIGHTS.get(agent, 0)
    print(f"  #{rank} {agent:<22} Combined={ca:.3f}  RQ={rq:.1f}/10  w={wt:.3f}")
print()
if 'baseline_both_correct' in panda_df.columns:
    base_both = panda_df['baseline_both_correct'].mean()
    cons_both = panda_df['consensus_both_correct'].mean()
    print(f"  PANDA ACCURACY:")
    print(f"    Pre-debate (R1 weighted) : {base_both:.3f}")
    print(f"    Post-debate (R3 weighted): {cons_both:.3f}  Δ={cons_both-base_both:+.3f}")
print()
top_p = max(debate_agents, key=lambda n: sum(influence_matrix.get((n,o),0) for o in debate_agents if o!=n))
print(f"  TOP PERSUADER    : {top_p}")
most_stubborn = max(debate_agents, key=lambda n: stubbornness[n])
print(f"  MOST STUBBORN    : {most_stubborn} ({stubbornness[most_stubborn]/n_img:.1%})")
print()
print("  OUTPUT FILES:")
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print(f"    {p.relative_to(OUTPUT_DIR)} ({p.stat().st_size/1024:.1f} KB)")
print("="*70)